# Evaluation and figures (RQ1-RQ4)

This notebook displays the outputs of `run.sh evaluate` (primary evaluation: coverage, binding yield, acceptance tests), `run.sh figures` and `run.sh analyze` (secondary analyses: hardness factors, capabilities, reconciliation, optimality gaps, HSC reference quality). It computes nothing itself.

* **RQ1 - What makes a QFBS instance hard?** `secondary/factor_analysis.csv`, `factor_effects.csv`, `figures/fig_rq1_hardness`.
* **RQ2 - Which instances can each solver process, and how often does it return a binding?** `evaluation/outcomes_by_suite_engine.csv`, `figures/rq2_engine_coverage`, `secondary/capability_matrix.csv`, `incompatibilities.csv`, `termination_reasons.csv`.
* **RQ3 - How do outcomes differ between the QFBS feasibility settings, and do the solvers' verdicts agree per instance?** `evaluation/qfbs_outcomes.csv`, `secondary/reconciliation_summary.csv`, `contradictions.csv`, `figures/fig_rq3_reconciliation`.
* **RQ4 - How close are heuristic bindings to certified optima (and to the authors' reference compositions in HSC)?** `secondary/optimality_gap.csv`, `vs_cp_incumbent.csv`, `hsc_quality_summary.json`, `figures/fig_rq4_gap`, `fig_rq4_gap_recovered`, `fig_rq4_hsc`.


In [ ]:
from pathlib import Path
import csv, json
import pandas as pd
from IPython.display import display, SVG, Markdown
ROOT = Path.cwd()
ART = ROOT / ".artifacts/qacobench"
EVAL, FIG, SEC = ART / "evaluation", ART / "figures", ART / "secondary"
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 40)

def show_svg(path):
    display(SVG(filename=str(path))) if Path(path).is_file() else print("missing", path)

def table(path, **kw):
    return pd.read_csv(path, **kw) if Path(path).is_file() else print("missing", path)


## Acceptance of the campaign

The evaluation accepts a campaign only when the release counts match, no run is duplicated, no infrastructure error occurred, no guaranteed-feasible instance is labelled UNSAT, and the exact labels survive reconciliation (`unsat_with_validated_binding = 0`, `optimal_beaten_by_other_engine = 0`).

In [ ]:
print(json.dumps(json.loads((EVAL / "acceptance.json").read_text()), indent=2))
print(json.dumps(json.loads((EVAL / "reconciliation_failures.json").read_text()), indent=2)[:2000])

## RQ1 - What makes a QFBS instance hard?

Marginal outcome rates per design-factor level: share of instances the exact solver settles within 5 s (OPTIMAL or UNSAT), median proof time, unresolved instances, and the share of certified optima the GA attains. Factors are ordered by the spread of the proof rate across their levels.

In [ ]:
display(table(SEC / "factor_effects.csv"))
show_svg(FIG.parent / "secondary/figures/fig_rq1_hardness.svg")
display(table(SEC / "factor_analysis.csv")[["factor", "level", "instances", "MZN_proof_rate_all", "MZN_median_proof_ms", "unresolved_rate", "GA_share_matches_optimum", "GA_mean_gap_on_OPTIMAL"]])

## RQ2 - Solver applicability: coverage and binding yield

Coverage = processable / submitted; conditional binding yield = binding returned / processable. Rejections are recorded as INCOMPATIBLE with the engine's own diagnostic, aggregated in the capability matrix; termination reasons separate proofs, evaluation budgets and the time budget.

In [ ]:
show_svg(FIG / "rq2_engine_coverage.svg")
display(table(EVAL / "outcomes_by_suite_engine.csv"))
display(table(SEC / "capability_matrix.csv"))
display(table(EVAL / "termination_reasons.csv"))

## RQ3 - Outcomes under the two feasibility settings and cross-solver agreement

Per-instance reconciliation of the exact labels with the heuristics' gateway-validated bindings. `contradictions.csv` must be empty for the labels to be published.

In [ ]:
display(table(EVAL / "qfbs_outcomes.csv"))
show_svg(FIG.parent / "secondary/figures/fig_rq3_reconciliation.svg")
display(table(SEC / "reconciliation_summary.csv"))
display(table(SEC / "contradictions.csv"))

## RQ4 - Solution quality against certified optima and reference compositions

Gap = heuristic score - MiniZinc optimum (same normalised weighted sum for every engine). For HSC, which has no exact proofs, the scores are compared with the authors' reference compositions (Eq. 1 + Eq. 2 recomputed on the released data).

In [ ]:
show_svg(FIG.parent / "secondary/figures/fig_rq4_gap.svg")
show_svg(FIG.parent / "secondary/figures/fig_rq4_gap_recovered.svg")
g = table(SEC / "optimality_gap.csv")
display(g[["by", "level", "engine", "instances", "with_binding", "share_matches_optimum_of_bound", "gap_mean", "gap_p90", "gap_max"]])
display(table(SEC / "vs_cp_incumbent.csv"))
show_svg(FIG.parent / "secondary/figures/fig_rq4_hsc.svg")
print(json.dumps(json.loads((SEC / "hsc_quality_summary.json").read_text()), indent=1)[:3000])
display(table(SEC / "hsc_quality_by_constraints.csv"))

## Timing and budgets

Runtimes, termination reasons and time to the first feasible binding, per suite and engine.

In [ ]:
display(table(SEC / "timing_budget.csv"))
show_svg(FIG.parent / "secondary/figures/fig_time_to_binding.svg")
if (SEC / "repeat_variability.csv").is_file():
    display(table(SEC / "repeat_variability.csv"))